# Data Preparation: ZINC15 Dataset

This notebook explores the ZINC15 drug-like subset, calculates molecular properties,
and prepares the data for training molecular generation models.

## Contents
1. Load and explore ZINC15 data
2. Calculate molecular properties
3. Visualize property distributions
4. Analyze cancer drug constraints
5. Data quality assessment

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Draw

# Add src to path
sys.path.insert(0, '../src')
from property_calculator import calculate_all_properties, check_cancer_drug_constraints

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

Load the training and validation datasets.

In [ ]:
# Load datasets
df_train = pd.read_csv('../data/zinc15_train.csv')
df_val = pd.read_csv('../data/zinc15_val.csv')

print(f"Training set: {len(df_train)} molecules")
print(f"Validation set: {len(df_val)} molecules")
print(f"Total: {len(df_train) + len(df_val)} molecules")

# Display first few molecules
print("\nFirst 5 training molecules:")
df_train.head()

## 2. Calculate Molecular Properties

Calculate properties for all molecules in the training set.

In [ ]:
# Calculate properties for training set
print("Calculating properties for training set...")
properties_list = []

for smi in tqdm(df_train['smiles']):
    props = calculate_all_properties(smi)
    if props:
        properties_list.append(props)

df_props = pd.DataFrame(properties_list)
print(f"\nCalculated properties for {len(df_props)} molecules")
df_props.head()

## 3. Property Statistics

Analyze statistical properties of the dataset.

In [ ]:
# Summary statistics
print("Property Statistics:")
print("=" * 60)
print(df_props[['molecular_weight', 'logp', 'hbd', 'hba', 'psa', 
                'aromatic_rings', 'rotatable_bonds', 'sa_score', 'qed']].describe())

## 4. Visualize Property Distributions

Create histograms for key molecular properties.

In [ ]:
# Property distribution plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Molecular Property Distributions', fontsize=16, y=0.995)

properties_to_plot = [
    ('molecular_weight', 'Molecular Weight (Da)', 20),
    ('logp', 'LogP', 20),
    ('hbd', 'Hydrogen Bond Donors', 10),
    ('hba', 'Hydrogen Bond Acceptors', 10),
    ('psa', 'Polar Surface Area (Ų)', 20),
    ('aromatic_rings', 'Aromatic Rings', 10),
    ('rotatable_bonds', 'Rotatable Bonds', 10),
    ('sa_score', 'SA Score', 20),
    ('qed', 'QED (Drug-likeness)', 20),
]

for idx, (prop, label, bins) in enumerate(properties_to_plot):
    ax = axes[idx // 3, idx % 3]
    ax.hist(df_props[prop], bins=bins, edgecolor='black', alpha=0.7)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'{label} Distribution')
    
    # Add mean line
    mean_val = df_props[prop].mean()
    ax.axvline(mean_val, color='r', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/property_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Cancer Drug Constraint Analysis

Check how many molecules meet cancer drug fragment constraints:
- MW < 300 Da
- LogP between 1.5-4.0
- HBD ≤ 5
- HBA ≤ 10
- PSA < 140 Ų
- Rotatable bonds ≤ 10

In [ ]:
# Check cancer drug constraints
print("Analyzing cancer drug constraints...")
constraint_results = []

for smi in tqdm(df_train['smiles']):
    passes, constraints, _ = check_cancer_drug_constraints(smi)
    constraints['overall_pass'] = passes
    constraints['smiles'] = smi
    constraint_results.append(constraints)

df_constraints = pd.DataFrame(constraint_results)

# Summary
print("\nCancer Drug Constraint Summary:")
print("=" * 60)
print(f"Molecules passing all constraints: {df_constraints['overall_pass'].sum()} / {len(df_constraints)} ({df_constraints['overall_pass'].mean()*100:.1f}%)")
print("\nIndividual constraint pass rates:")
for col in ['mw_check', 'logp_check', 'hbd_check', 'hba_check', 'psa_check', 'rotatable_bonds_check']:
    pass_rate = df_constraints[col].mean() * 100
    print(f"  {col}: {pass_rate:.1f}%")

In [ ]:
# Visualize constraint pass rates
fig, ax = plt.subplots(figsize=(10, 6))

constraints_labels = {
    'mw_check': 'MW < 300',
    'logp_check': 'LogP 1.5-4.0',
    'hbd_check': 'HBD ≤ 5',
    'hba_check': 'HBA ≤ 10',
    'psa_check': 'PSA < 140',
    'rotatable_bonds_check': 'Rot. Bonds ≤ 10',
    'overall_pass': 'All Constraints'
}

constraints = ['mw_check', 'logp_check', 'hbd_check', 'hba_check', 
               'psa_check', 'rotatable_bonds_check', 'overall_pass']
pass_rates = [df_constraints[c].mean() * 100 for c in constraints]
labels = [constraints_labels[c] for c in constraints]

colors = ['green' if c != 'overall_pass' else 'blue' for c in constraints]
ax.barh(labels, pass_rates, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Pass Rate (%)')
ax.set_title('Cancer Drug Constraint Pass Rates')
ax.set_xlim(0, 100)

# Add percentage labels
for i, v in enumerate(pass_rates):
    ax.text(v + 2, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig('../results/figures/constraint_pass_rates.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Visualize Example Molecules

Display structures of some example molecules.

In [ ]:
# Visualize a few example molecules
n_examples = min(12, len(df_train))
example_smiles = df_train['smiles'].sample(n=n_examples, random_state=42).tolist()
example_mols = [Chem.MolFromSmiles(smi) for smi in example_smiles]

# Create grid of molecules
img = Draw.MolsToGridImage(example_mols, molsPerRow=4, subImgSize=(300, 300),
                           legends=[f"Mol {i+1}" for i in range(n_examples)])

# Save and display
img.save('../results/figures/example_molecules.png')
img

## 7. Property Correlations

Analyze correlations between different molecular properties.

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))

props_for_corr = ['molecular_weight', 'logp', 'hbd', 'hba', 'psa', 
                  'aromatic_rings', 'rotatable_bonds', 'qed']
corr_matrix = df_props[props_for_corr].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Molecular Property Correlations')
plt.tight_layout()
plt.savefig('../results/figures/property_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Data Quality Summary

Final summary of data quality and statistics.

In [ ]:
print("="*70)
print("DATA QUALITY SUMMARY")
print("="*70)

print(f"\nDataset Size:")
print(f"  Training: {len(df_train)} molecules")
print(f"  Validation: {len(df_val)} molecules")
print(f"  Total: {len(df_train) + len(df_val)} molecules")

print(f"\nProperty Ranges:")
print(f"  Molecular Weight: {df_props['molecular_weight'].min():.1f} - {df_props['molecular_weight'].max():.1f} Da")
print(f"  LogP: {df_props['logp'].min():.2f} - {df_props['logp'].max():.2f}")
print(f"  HBD: {int(df_props['hbd'].min())} - {int(df_props['hbd'].max())}")
print(f"  HBA: {int(df_props['hba'].min())} - {int(df_props['hba'].max())}")
print(f"  PSA: {df_props['psa'].min():.1f} - {df_props['psa'].max():.1f} Ų")
print(f"  QED: {df_props['qed'].min():.3f} - {df_props['qed'].max():.3f}")

print(f"\nDrug-likeness:")
print(f"  Mean QED: {df_props['qed'].mean():.3f}")
print(f"  Molecules with QED > 0.5: {(df_props['qed'] > 0.5).sum()} ({(df_props['qed'] > 0.5).mean()*100:.1f}%)")

print(f"\nCancer Drug Constraints:")
print(f"  Molecules passing all constraints: {df_constraints['overall_pass'].sum()} ({df_constraints['overall_pass'].mean()*100:.1f}%)")

print(f"\nData Quality: {'GOOD' if len(df_train) > 0 else 'POOR'}")
print("="*70)

## 9. Export Property Statistics

Save property statistics for future reference.

In [ ]:
# Save property statistics
df_props.to_csv('../data/molecule_properties.csv', index=False)
print("Saved property statistics to ../data/molecule_properties.csv")

# Save constraint results
df_constraints.to_csv('../data/constraint_results.csv', index=False)
print("Saved constraint results to ../data/constraint_results.csv")

print("\nData preparation complete!")